# 009-v9recall 最小 ICE forward（11分子/12支持对，Colab T4）。按顺序跑，每格末尾有 EXPECT，对不上就停、贴回来。
只含11个研究分子（NP 8 + G1 3），无保留集，不调参。truth仅本地离线打分用。预计实跑5-10分钟，保持页面在线。
3个文件（`v9recall-bundle.tar.gz` + `gen.pt` + `inten.pt`）放 Drive 任意位置即可：第2格会自动搜，不用改路径。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# EXPECT: Mounted at /content/drive
# 备选：不用 Drive 则跳过本格，把3个文件用左侧文件面板直接上传到 /content，第2格照样能找到。

In [ ]:
import os
SEARCH_DIRS = ["/content/drive/MyDrive/v9recall-weights", "/content/drive/MyDrive", "/content"]
NEED = ["v9recall-bundle.tar.gz", "gen.pt", "inten.pt"]
found = {}
for name in NEED:
    hit = next((d + "/" + name for d in SEARCH_DIRS if os.path.isfile(d + "/" + name)), None)
    found[name] = hit
    print(name, "->", hit if hit else "NOT FOUND")
print("searched:", SEARCH_DIRS)
missing = [k for k, v in found.items() if v is None]
assert not missing, "找不到: %s。请把这几个文件传到上面任一目录（Drive根目录最省事）后重跑本格。" % missing
os.environ["V9_TAR"] = found["v9recall-bundle.tar.gz"]
os.environ["V9_GEN"] = found["gen.pt"]
os.environ["V9_INTEN"] = found["inten.pt"]
# EXPECT: 三行全部是具体路径（不是 NOT FOUND），无 AssertionError。后面的格都用这里定位到的路径，不再猜。

In [ ]:
import os, shutil
tar = os.environ["V9_TAR"]
shutil.rmtree("/content/v9recall-bundle", ignore_errors=True)
os.makedirs("/content/v9recall-bundle", exist_ok=True)
# 注意：本包tar是flat结构（顶层即inputs.json，无v9recall-bundle/前缀），必须 -C 进工作目录解压
!tar xzf "{tar}" -C /content/v9recall-bundle && test -f /content/v9recall-bundle/inputs.json && ls /content/v9recall-bundle
# EXPECT: 看到 inputs.json install.sh g1_worker.py MANIFEST.iceberg.json new_candidates_12.json weights_reference.json README.md code/

In [ ]:
import os, hashlib, shutil
os.makedirs("/content/v9recall-bundle/weights", exist_ok=True)
shutil.copy(os.environ["V9_GEN"], "/content/v9recall-bundle/weights/gen.pt")
shutil.copy(os.environ["V9_INTEN"], "/content/v9recall-bundle/weights/inten.pt")
!ls -l /content/v9recall-bundle/weights/
EXP = {"gen.pt": ("deef0451c1870d2125653bb212c9d95c5b5d9bf80156dfa9b22fedda7db49bea", 14561244),
       "inten.pt": ("c49e330e049d3a8b0095026ed694fcf70045c24305354853732aaa5f89d55068", 13793037)}
for name, (sha, size) in EXP.items():
    p = "/content/v9recall-bundle/weights/" + name
    assert os.path.getsize(p) == size, "%s bytes mismatch" % name
    h = hashlib.sha256(open(p, "rb").read()).hexdigest()
    assert h == sha, "%s sha mismatch: %s" % (name, h)
    print(name, "OK", size, sha[:12])
# EXPECT: gen.pt OK / inten.pt OK，无 AssertionError。mismatch 必须停，不要继续跑。

In [ ]:
import torch
assert torch.cuda.is_available(), "本格必须 T4 GPU 运行时：改 运行时→更改运行时类型→T4 后重跑"
print(torch.cuda.get_device_name(0))
# EXPECT: Tesla T4

In [ ]:
!cd /content/v9recall-bundle && bash install.sh
# EXPECT: 末尾 install done. 约2-4分钟。

In [ ]:
!cd /content/v9recall-bundle && PYTHONPATH=code/ice/shim:code/ice/src python3 g1_worker.py --pkg code/ice --model iceberg --inputs inputs.json --out /tmp/dry.json --k 100 --cemode split --dry-run --mol-limit 2
# EXPECT: DRY-RUN: 2 mols, 436 unique jobs，status ok-dry-run，无 Traceback。只验接线，不跑模型。

In [ ]:
!cd /content/v9recall-bundle && mkdir -p scores cache work work/ice_ckpt_pkg/ckpt && ln -sf $(pwd)/weights/gen.pt work/ice_ckpt_pkg/ckpt/gen.pt && ln -sf $(pwd)/weights/inten.pt work/ice_ckpt_pkg/ckpt/inten.pt && ls -l work/ice_ckpt_pkg/ckpt/ && PYTHONPATH=code/ice/shim:code/ice/src python3 g1_worker.py --pkg code/ice --model iceberg --inputs inputs.json --out scores/v9recall_ice_top100_split.json --k 100 --cemode split --device cuda --cache-dir cache --workdir work --ckpt-pkg work/ice_ckpt_pkg --any-rdkit --chunk-jobs 3000
# EXPECT: 末尾 status=ok（不是ok-dry-run）。约5-10分钟，meta 含 pred_per_s/max_mem_gb。增量41 jobs秒级预测，大头是模型加载。

In [ ]:
# 可选 mean 对照（复用缓存，增量小；不需要可跳过整格）
!cd /content/v9recall-bundle && PYTHONPATH=code/ice/shim:code/ice/src python3 g1_worker.py --pkg code/ice --model iceberg --inputs inputs.json --out scores/v9recall_ice_top100_mean.json --k 100 --cemode mean --device cuda --cache-dir cache --workdir work --ckpt-pkg work/ice_ckpt_pkg --any-rdkit
# EXPECT: status=ok。跳过不影响主结果回传。

In [ ]:
!cd /content/v9recall-bundle && ls -lh scores/ && tar czf v9recall-scores.tar.gz scores && ls -lh v9recall-scores.tar.gz
# EXPECT: scores/ 下有 v9recall_ice_top100_split.json(+.meta.json)；v9recall-scores.tar.gz 若干MB。去左侧 Files 面板下载发回（+可选 cache/*.npz）。